# Stable Diffusion v1.5 Porting to LiteRT (TFLite)

이 노트북은 HuggingFace의 Stable Diffusion v1.5 모델을 LiteRT Edge Generative API를 사용하여
TFLite 모델로 변환하고, 모바일 디바이스에서 실행 가능한 형태로 포팅하는 전체 과정을 보여줍니다.

## 전체 워크플로우

1. **환경 설정** - 필요한 패키지 설치
2. **체크포인트 다운로드** - HuggingFace에서 SD 1.5 가중치 다운로드
3. **모델 재작성 (Re-author)** - Edge Generative API 빌딩 블록으로 모델 구성
4. **가중치 로드** - 원본 체크포인트에서 재작성 모델로 가중치 매핑
5. **변환 (Convert)** - PyTorch → TFLite 변환
6. **양자화 (Quantize)** - 모델 크기 및 성능 최적화
7. **추론 (Inference)** - 변환된 TFLite 모델로 이미지 생성

## 1. 환경 설정

In [ ]:
# 필수 패키지 설치
# litert-torch는 LiteRT 변환 및 Edge Generative API를 포함합니다.
!pip install -e ./litert-torch
!pip install safetensors pillow tqdm numpy regex
!pip install huggingface-hub[xet] transformers tensorflow
!pip install --upgrade ai_edge_litert ai_edge_quantizer

In [ ]:
import os
import pathlib

import litert_torch
import litert_torch.generative.layers.builder as layers_builder
import litert_torch.generative.layers.model_config as layers_cfg
from litert_torch.generative.layers.unet import blocks_2d
import litert_torch.generative.layers.unet.model_config as unet_cfg
from litert_torch.generative.layers.attention import TransformerBlock
import litert_torch.generative.layers.attention_utils as attention_utils
from litert_torch.generative.quantize import quant_recipes
from litert_torch.generative.utilities import stable_diffusion_loader
import litert_torch.generative.utilities.loader as loading_utils
import numpy as np
from PIL import Image
import torch
from torch import nn
import tqdm

print(f"LiteRT Torch version: {litert_torch.__version__}")
print(f"PyTorch version: {torch.__version__}")

## 2. 체크포인트 다운로드

HuggingFace에서 Stable Diffusion v1.5의 safetensors 체크포인트를 다운로드합니다.

사전에 `huggingface-cli`로 로그인하거나, 수동으로 다운로드하여 경로를 설정하세요.

In [ ]:
# 체크포인트 경로 설정
# HuggingFace에서 stable-diffusion-v1-5 리포를 클론하거나
# v1-5-pruned-emaonly.safetensors 파일을 다운로드하세요.
#
# 방법 1: huggingface-cli 사용
# !huggingface-cli download stable-diffusion-v1-5/stable-diffusion-v1-5 \
#     v1-5-pruned-emaonly.safetensors \
#     --local-dir ~/stable-diffusion-v1-5/
#
# 방법 2: git lfs 사용
# !git lfs install
# !git clone https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5 ~/stable-diffusion-v1-5

CKPT_DIR = os.path.join(pathlib.Path.home(), "stable-diffusion-v1-5")
CKPT_PATH = os.path.join(CKPT_DIR, "v1-5-pruned-emaonly.safetensors")
TOKENIZER_DIR = os.path.join(CKPT_DIR, "tokenizer")
OUTPUT_DIR = "/tmp/sd15_tflite"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Checkpoint path: {CKPT_PATH}")
print(f"Checkpoint exists: {os.path.exists(CKPT_PATH)}")
print(f"Tokenizer dir: {TOKENIZER_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

## 3. SD 1.5 아키텍처 이해

Stable Diffusion v1.5는 4개의 핵심 컴포넌트로 구성됩니다:

| 컴포넌트 | 역할 | 입력 | 출력 |
|---------|------|------|------|
| **CLIP Text Encoder** | 텍스트 → 임베딩 | 토큰 (77,) | 컨텍스트 (1, 77, 768) |
| **Diffusion (UNet)** | 노이즈 제거 | 잠재 텐서 + 컨텍스트 + 타임스텝 | 노이즈 예측 |
| **VAE Decoder** | 잠재 공간 → 이미지 | 잠재 텐서 (1, 4, 64, 64) | 이미지 (1, 3, 512, 512) |
| **VAE Encoder** | 이미지 → 잠재 공간 | 이미지 (1, 3, 512, 512) | 잠재 텐서 (1, 4, 64, 64) |

각 컴포넌트를 Edge Generative API의 빌딩 블록으로 재작성합니다.

## 4. 모델 재작성 (Re-authoring)

Edge Generative API에서 제공하는 레이어를 사용하여 각 컴포넌트를 재작성합니다.

### 사용하는 빌딩 블록:
- `TransformerBlock` - CLIP의 트랜스포머 블록
- `blocks_2d.DownEncoderBlock2D` - UNet 다운 인코더
- `blocks_2d.MidBlock2D` - UNet 미드 블록
- `blocks_2d.SkipUpDecoderBlock2D` - UNet 업 디코더 (스킵 커넥션)
- `blocks_2d.UpDecoderBlock2D` - VAE 디코더의 업 디코더

### 4.1 CLIP Text Encoder

CLIP은 텍스트 프롬프트를 768차원 임베딩으로 변환합니다.
12개의 Transformer 블록, LayerNorm, GELU_QUICK 활성화를 사용합니다.

In [ ]:
# CLIP 텐서 이름 매핑
# 원본 HF 체크포인트의 키 이름을 재작성 모델에 매핑합니다.
CLIP_TENSOR_NAMES = loading_utils.ModelLoader.TensorNames(
    ff_up_proj="cond_stage_model.transformer.text_model.encoder.layers.{}.mlp.fc1",
    ff_down_proj="cond_stage_model.transformer.text_model.encoder.layers.{}.mlp.fc2",
    attn_query_proj="cond_stage_model.transformer.text_model.encoder.layers.{}.self_attn.q_proj",
    attn_key_proj="cond_stage_model.transformer.text_model.encoder.layers.{}.self_attn.k_proj",
    attn_value_proj="cond_stage_model.transformer.text_model.encoder.layers.{}.self_attn.v_proj",
    attn_output_proj="cond_stage_model.transformer.text_model.encoder.layers.{}.self_attn.out_proj",
    pre_attn_norm="cond_stage_model.transformer.text_model.encoder.layers.{}.layer_norm1",
    post_attn_norm="cond_stage_model.transformer.text_model.encoder.layers.{}.layer_norm2",
    embedding="cond_stage_model.transformer.text_model.embeddings.token_embedding",
    embedding_position="cond_stage_model.transformer.text_model.embeddings.position_embedding.weight",
    final_norm="cond_stage_model.transformer.text_model.final_layer_norm",
    lm_head=None,
)

print("CLIP tensor names configured.")
print(f"  Attention Q proj pattern: {CLIP_TENSOR_NAMES.attn_query_proj}")

In [ ]:
class CLIP(nn.Module):
    """CLIP text encoder for Stable Diffusion v1.5.

    - 12 Transformer 블록
    - 768 차원 임베딩
    - 77 토큰 시퀀스 길이 (고정)
    - Causal Self-Attention
    """

    def __init__(self, config: layers_cfg.ModelConfig):
        super().__init__()
        self.tok_embedding = nn.Embedding(config.vocab_size, config.embedding_dim)
        self.tok_embedding_position = nn.Parameter(
            torch.zeros((config.max_seq_len, config.embedding_dim)),
            requires_grad=False,
        )
        self.config = config
        block_config = config.block_config(0)
        self.transformer_blocks = nn.ModuleList(
            TransformerBlock(block_config, config)
            for _ in range(config.num_layers)
        )
        self.final_norm = layers_builder.build_norm(
            config.embedding_dim, config.final_norm_config
        )
        self.mask_cache = attention_utils.build_causal_mask_cache(
            size=config.max_seq_len, dtype=torch.float32
        )

    @torch.inference_mode
    def forward(self, tokens: torch.IntTensor) -> torch.FloatTensor:
        state = self.tok_embedding(tokens) + self.tok_embedding_position
        for layer in self.transformer_blocks:
            state = layer(state, mask=self.mask_cache)
        output = self.final_norm(state)
        return output


def get_clip_config() -> layers_cfg.ModelConfig:
    """CLIP 모델 설정 (SD v1.5)."""
    attn_config = layers_cfg.AttentionConfig(
        num_heads=12,
        head_dim=64,  # 768 // 12
        num_query_groups=12,
        rotary_base=0,
        rotary_percentage=0.0,
        qkv_use_bias=True,
        qkv_transpose_before_split=True,
        qkv_fused_interleaved=False,
        output_proj_use_bias=True,
        enable_kv_cache=False,
    )
    ff_config = layers_cfg.FeedForwardConfig(
        type=layers_cfg.FeedForwardType.SEQUENTIAL,
        activation=layers_cfg.ActivationConfig(layers_cfg.ActivationType.GELU_QUICK),
        intermediate_size=768 * 4,  # 3072
        use_bias=True,
    )
    norm_config = layers_cfg.NormalizationConfig(
        type=layers_cfg.NormalizationType.LAYER_NORM, enable_hlfb=False
    )
    block_config = layers_cfg.TransformerBlockConfig(
        attn_config=attn_config,
        ff_config=ff_config,
        pre_attention_norm_config=norm_config,
        post_attention_norm_config=norm_config,
    )
    return layers_cfg.ModelConfig(
        vocab_size=49408,
        num_layers=12,
        max_seq_len=77,
        embedding_dim=768,
        block_configs=block_config,
        final_norm_config=norm_config,
    )


print("CLIP model class defined.")
print(f"  vocab_size=49408, num_layers=12, embedding_dim=768, max_seq_len=77")

### 4.2 Diffusion Model (UNet)

UNet 기반의 Diffusion 모델은 SD 1.5의 핵심입니다.

```
latents ─► ConvIn ─► DownEncoder(x4) ─► MidBlock ─► SkipUpDecoder(x4) ─► Norm ─► Act ─► ConvOut
                         │                                    ▲
                         └──── skip connections ──────────────┘
```

주요 설정:
- `block_out_channels = [320, 640, 1280, 1280]`
- 8 attention heads
- Cross-attention dim = 768 (CLIP 임베딩 크기)
- GroupNorm (32 그룹), SiLU 활성화

In [ ]:
# 기존 구현을 직접 임포트합니다.
# Edge Generative API는 이미 SD 1.5 예제를 포함하고 있으므로
# 이를 활용합니다.
from litert_torch.generative.examples.stable_diffusion import clip
from litert_torch.generative.examples.stable_diffusion import diffusion
from litert_torch.generative.examples.stable_diffusion import decoder
from litert_torch.generative.examples.stable_diffusion import util
from litert_torch.generative.examples.stable_diffusion import tokenizer
from litert_torch.generative.examples.stable_diffusion import samplers

print("All SD 1.5 modules imported successfully.")

### 4.3 모델 설정 확인

각 컴포넌트의 설정을 확인합니다.

In [ ]:
# CLIP 설정
clip_config = clip.get_model_config()
print("=== CLIP Config ===")
print(f"  vocab_size: {clip_config.vocab_size}")
print(f"  num_layers: {clip_config.num_layers}")
print(f"  embedding_dim: {clip_config.embedding_dim}")
print(f"  max_seq_len: {clip_config.max_seq_len}")
print()

# Diffusion (UNet) 설정
diff_config = diffusion.get_model_config(batch_size=2)
print("=== Diffusion (UNet) Config ===")
print(f"  in_channels: {diff_config.in_channels}")
print(f"  out_channels: {diff_config.out_channels}")
print(f"  block_out_channels: {diff_config.block_out_channels}")
print(f"  layers_per_block: {diff_config.layers_per_block}")
print(f"  transformer_num_attention_heads: {diff_config.transformer_num_attention_heads}")
print(f"  transformer_cross_attention_dim: {diff_config.transformer_cross_attention_dim}")
print()

# Decoder (VAE) 설정
dec_config = decoder.get_model_config()
print("=== Decoder (VAE) Config ===")
print(f"  latent_channels: {dec_config.latent_channels}")
print(f"  out_channels: {dec_config.out_channels}")
print(f"  block_out_channels: {dec_config.block_out_channels}")
print(f"  scaling_factor: {dec_config.scaling_factor}")
print(f"  layers_per_block: {dec_config.layers_per_block}")

## 5. 가중치 로드 및 모델 구성

원본 safetensors 체크포인트에서 가중치를 로드합니다.
`stable_diffusion_loader`가 텐서 이름 매핑을 자동으로 처리합니다.

In [ ]:
@torch.inference_mode()
def build_clip_model(ckpt_path: str) -> nn.Module:
    """CLIP 모델을 생성하고 가중치를 로드합니다."""
    model = clip.CLIP(clip.get_model_config())
    loader = stable_diffusion_loader.ClipModelLoader(
        ckpt_path, clip.TENSOR_NAMES
    )
    loader.load(model, strict=False)
    model.eval()
    print(f"CLIP model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model


@torch.inference_mode()
def build_diffusion_model(ckpt_path: str, batch_size: int = 2) -> nn.Module:
    """Diffusion (UNet) 모델을 생성하고 가중치를 로드합니다."""
    model = diffusion.Diffusion(
        diffusion.get_model_config(batch_size=batch_size)
    )
    loader = stable_diffusion_loader.DiffusionModelLoader(
        ckpt_path, diffusion.TENSOR_NAMES
    )
    loader.load(model, strict=False)
    model.eval()
    print(f"Diffusion model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model


@torch.inference_mode()
def build_decoder_model(ckpt_path: str) -> nn.Module:
    """VAE Decoder 모델을 생성하고 가중치를 로드합니다."""
    model = decoder.Decoder(decoder.get_model_config())
    loader = stable_diffusion_loader.AutoEncoderModelLoader(
        ckpt_path, decoder.TENSOR_NAMES
    )
    loader.load(model, strict=False)
    model.eval()
    print(f"Decoder model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model


print("Model builder functions defined.")

In [ ]:
# 모델 로드 (체크포인트가 있을 때만 실행)
if os.path.exists(CKPT_PATH):
    clip_model = build_clip_model(CKPT_PATH)
    diffusion_model = build_diffusion_model(CKPT_PATH)
    decoder_model = build_decoder_model(CKPT_PATH)
    print("\nAll models loaded successfully!")
else:
    print(f"Checkpoint not found at: {CKPT_PATH}")
    print("Please download the checkpoint first (see Section 2).")

## 6. TFLite 변환 (Convert)

`litert_torch.signature()` API를 사용하여 각 컴포넌트를 별도의 TFLite 파일로 변환합니다.

### 변환 흐름:
1. 트레이싱용 더미 입력 생성
2. `litert_torch.signature(name, model, args).convert()` 호출
3. `.export(path)` 로 TFLite 파일 저장

In [ ]:
IMAGE_HEIGHT = 512
IMAGE_WIDTH = 512
N_TOKENS = 77


@torch.inference_mode()
def convert_all_to_tflite(
    clip_model: nn.Module,
    diffusion_model: nn.Module,
    decoder_model: nn.Module,
    output_dir: str,
    quantize: bool = False,
):
    """모든 SD 1.5 컴포넌트를 TFLite로 변환합니다."""

    # 양자화 설정
    quant_config = quant_recipes.full_weight_only_recipe() if quantize else None
    if quantize:
        print("Quantization enabled: weight-only INT8")

    # --- 1. CLIP 변환 ---
    print("\n[1/3] Converting CLIP text encoder...")
    prompt_tokens = torch.full((1, N_TOKENS), 0, dtype=torch.int)

    litert_torch.signature(
        "encode", clip_model, (prompt_tokens,)
    ).convert(
        quant_config=quant_config
    ).export(
        f"{output_dir}/clip.tflite"
    )
    print(f"  Saved: {output_dir}/clip.tflite")

    # --- 2. Diffusion (UNet) 변환 ---
    print("\n[2/3] Converting Diffusion (UNet) model...")
    noise = torch.zeros(
        (1, 4, IMAGE_HEIGHT // 8, IMAGE_WIDTH // 8), dtype=torch.float32
    )
    input_latents = torch.zeros_like(noise)

    # CLIP을 통해 context 형태를 맞춤
    context_cond = clip_model(prompt_tokens)
    context_uncond = torch.zeros_like(context_cond)
    context = torch.cat([context_cond, context_uncond], axis=0)

    time_embedding = util.get_time_embedding(0)

    litert_torch.signature(
        "diffusion",
        diffusion_model,
        (torch.repeat_interleave(input_latents, 2, 0), context, time_embedding),
    ).convert(
        quant_config=quant_config
    ).export(
        f"{output_dir}/diffusion.tflite"
    )
    print(f"  Saved: {output_dir}/diffusion.tflite")

    # --- 3. VAE Decoder 변환 ---
    print("\n[3/3] Converting VAE Decoder...")
    litert_torch.signature(
        "decode", decoder_model, (input_latents,)
    ).convert(
        quant_config=quant_config
    ).export(
        f"{output_dir}/decoder.tflite"
    )
    print(f"  Saved: {output_dir}/decoder.tflite")

    print("\nConversion complete!")


print("Conversion function defined.")

In [ ]:
# 변환 실행 (양자화 없이)
if os.path.exists(CKPT_PATH):
    convert_all_to_tflite(
        clip_model,
        diffusion_model,
        decoder_model,
        output_dir=OUTPUT_DIR,
        quantize=False,  # True로 변경하면 weight-only INT8 양자화 적용
    )
else:
    print("Skipping conversion - checkpoint not available.")

In [ ]:
# 변환된 TFLite 파일 크기 확인
for name in ["clip", "diffusion", "decoder"]:
    path = f"{OUTPUT_DIR}/{name}.tflite"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{name}.tflite: {size_mb:.1f} MB")
    else:
        print(f"{name}.tflite: not found")

## 7. 양자화 (Quantization)

모바일 배포를 위해 양자화를 적용합니다. LiteRT는 다양한 양자화 레시피를 제공합니다:

| 레시피 | 가중치 | 연산 | 특징 |
|--------|--------|------|------|
| `full_int8_dynamic_recipe()` | INT8 | INT | 최소 크기 |
| `full_weight_only_recipe()` | INT8 | FP | 안정적 품질 |
| `full_fp16_recipe()` | FP16 | FP | 최고 품질 |
| `full_int4_dynamic_block32_recipe()` | INT4 | INT | 극한 압축 |

In [ ]:
# 양자화 적용 변환 (weight-only INT8)
QUANTIZED_OUTPUT_DIR = "/tmp/sd15_tflite_quantized"
os.makedirs(QUANTIZED_OUTPUT_DIR, exist_ok=True)

if os.path.exists(CKPT_PATH):
    convert_all_to_tflite(
        clip_model,
        diffusion_model,
        decoder_model,
        output_dir=QUANTIZED_OUTPUT_DIR,
        quantize=True,
    )

    # 크기 비교
    print("\n=== File Size Comparison ===")
    for name in ["clip", "diffusion", "decoder"]:
        orig = f"{OUTPUT_DIR}/{name}.tflite"
        quant = f"{QUANTIZED_OUTPUT_DIR}/{name}.tflite"
        if os.path.exists(orig) and os.path.exists(quant):
            orig_mb = os.path.getsize(orig) / (1024 * 1024)
            quant_mb = os.path.getsize(quant) / (1024 * 1024)
            ratio = quant_mb / orig_mb * 100
            print(f"{name}: {orig_mb:.1f} MB → {quant_mb:.1f} MB ({ratio:.0f}%)")
else:
    print("Skipping quantized conversion - checkpoint not available.")

## 8. TFLite 모델로 추론 (Inference)

변환된 TFLite 모델을 사용하여 이미지를 생성합니다.

### 추론 파이프라인:
1. **Tokenize** - 프롬프트를 토큰으로 변환
2. **CLIP Encode** - 토큰 → 텍스트 임베딩
3. **Diffusion Loop** - N 스텝 반복하여 노이즈 제거
4. **VAE Decode** - 잠재 텐서 → 이미지

In [ ]:
from ai_edge_litert import interpreter as interpreter_lib


class StableDiffusionTFLite:
    """TFLite 모델 기반 Stable Diffusion 파이프라인."""

    def __init__(
        self,
        tokenizer_vocab_dir: str,
        clip_tflite_path: str,
        diffusion_tflite_path: str,
        decoder_tflite_path: str,
        num_threads: int = os.cpu_count(),
    ):
        self.tokenizer = tokenizer.Tokenizer(tokenizer_vocab_dir)
        self.clip = litert_torch.load(clip_tflite_path)
        self.diffusion = litert_torch.load(diffusion_tflite_path)
        self.decoder = litert_torch.load(decoder_tflite_path)

        # 멀티스레드 인터프리터 설정
        for model, path in [
            (self.clip, clip_tflite_path),
            (self.diffusion, diffusion_tflite_path),
            (self.decoder, decoder_tflite_path),
        ]:
            model_bytes = model.model_content()
            model.set_interpreter_builder(
                lambda b=model_bytes: interpreter_lib.Interpreter(
                    model_content=b,
                    experimental_default_delegate_latest_features=True,
                    num_threads=num_threads,
                )
            )

        print(f"All TFLite models loaded. (num_threads={num_threads})")


def generate_image(
    model: StableDiffusionTFLite,
    prompt: str,
    output_path: str,
    uncond_prompt: str = "",
    cfg_scale: float = 7.5,
    height: int = 512,
    width: int = 512,
    sampler_name: str = "k_euler",
    n_inference_steps: int = 20,
    seed: int = None,
):
    """TFLite 모델로 이미지를 생성합니다."""

    if seed is not None:
        np.random.seed(seed)

    # Sampler 초기화
    if sampler_name == "k_euler":
        sampler = samplers.KEulerSampler(n_inference_steps=n_inference_steps)
    elif sampler_name == "k_euler_ancestral":
        sampler = samplers.KEulerAncestralSampler(n_inference_steps=n_inference_steps)
    elif sampler_name == "k_lms":
        sampler = samplers.KLMSSampler(n_inference_steps=n_inference_steps)
    else:
        raise ValueError(f"Unknown sampler: {sampler_name}")

    # Step 1: Text Encoding (CLIP)
    print("Step 1: Text encoding...")
    cond_tokens = model.tokenizer.encode(prompt)
    cond_context = model.clip(
        np.array(cond_tokens).astype(np.int32), signature_name="encode"
    )
    uncond_tokens = model.tokenizer.encode(uncond_prompt)
    uncond_context = model.clip(
        np.array(uncond_tokens).astype(np.int32), signature_name="encode"
    )
    context = np.concatenate([cond_context, uncond_context], axis=0)

    # Step 2: Initialize latents
    print("Step 2: Initializing latents...")
    noise_shape = (1, 4, height // 8, width // 8)
    latents = np.random.normal(size=noise_shape).astype(np.float32)
    latents *= sampler.initial_scale

    # Step 3: Diffusion loop
    print(f"Step 3: Diffusion ({n_inference_steps} steps)...")
    timesteps = tqdm.tqdm(sampler.timesteps, desc="Denoising")
    for _, timestep in enumerate(timesteps):
        time_embedding = util.get_time_embedding(timestep).numpy()

        input_latents = latents * sampler.get_input_scale()
        input_latents = input_latents.repeat(2, axis=0)

        output = model.diffusion(
            input_latents.astype(np.float32),
            context.astype(np.float32),
            time_embedding.astype(np.float32),
            signature_name="diffusion",
        )

        # Classifier-Free Guidance
        output_cond, output_uncond = np.split(output, 2, axis=0)
        output = cfg_scale * (output_cond - output_uncond) + output_uncond

        latents = sampler.step(latents, output)

    # Step 4: Decode latents to image
    print("Step 4: Decoding to image...")
    images = model.decoder(
        latents.astype(np.float32), signature_name="decode"
    )
    images = util.rescale(images, (-1, 1), (0, 255), clamp=True)
    images = util.move_channel(images, to="last")

    # 이미지 저장
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    result_image = Image.fromarray(images[0].astype(np.uint8))
    result_image.save(output_path)
    print(f"Image saved to: {output_path}")

    return result_image


print("Inference pipeline defined.")

In [ ]:
# TFLite 추론 실행
clip_tflite = f"{OUTPUT_DIR}/clip.tflite"
diffusion_tflite = f"{OUTPUT_DIR}/diffusion.tflite"
decoder_tflite = f"{OUTPUT_DIR}/decoder.tflite"

all_exist = all(
    os.path.exists(p) for p in [clip_tflite, diffusion_tflite, decoder_tflite]
)
tokenizer_exists = os.path.exists(TOKENIZER_DIR)

if all_exist and tokenizer_exists:
    sd_pipeline = StableDiffusionTFLite(
        tokenizer_vocab_dir=TOKENIZER_DIR,
        clip_tflite_path=clip_tflite,
        diffusion_tflite_path=diffusion_tflite,
        decoder_tflite_path=decoder_tflite,
    )

    result = generate_image(
        model=sd_pipeline,
        prompt="a photograph of an astronaut riding a horse",
        output_path=f"{OUTPUT_DIR}/generated_image.jpg",
        sampler_name="k_euler",
        n_inference_steps=20,
        seed=42,
    )
else:
    missing = []
    if not all_exist:
        missing.append("TFLite models")
    if not tokenizer_exists:
        missing.append(f"tokenizer dir ({TOKENIZER_DIR})")
    print(f"Skipping inference - missing: {', '.join(missing)}")

In [ ]:
# 생성된 이미지 표시
output_image_path = f"{OUTPUT_DIR}/generated_image.jpg"
if os.path.exists(output_image_path):
    from IPython.display import display
    display(Image.open(output_image_path))
else:
    print("No generated image found.")

## 9. 커스텀 모델 포팅 가이드

자체 Stable Diffusion 변형 모델을 포팅하려면 다음 단계를 따르세요:

### Step 1: 모델 구조 파악
```python
# 원본 모델의 state_dict 키를 확인
from safetensors.torch import load_file
state_dict = load_file("your_model.safetensors")
for key in sorted(state_dict.keys()):
    print(f"{key}: {state_dict[key].shape}")
```

### Step 2: 텐서 이름 매핑 작성
원본 체크포인트의 키 이름을 Edge Generative API 모델의 키 이름에 매핑합니다.

### Step 3: 모델 설정 작성
모델의 하이퍼파라미터를 `ModelConfig` / `DiffusionModelConfig` 등으로 정의합니다.

### Step 4: 변환 및 검증
```python
# 변환
edge_model = litert_torch.signature("sig_name", model, sample_args).convert()
edge_model.export("output.tflite")

# 검증 (원본 vs TFLite 출력 비교)
original_output = original_model(*sample_args)
tflite_output = edge_model(*sample_args, signature_name="sig_name")
print(f"Max diff: {np.max(np.abs(original_output - tflite_output))}")
```

## 10. 텐서 이름 매핑 상세

SD 1.5 포팅에서 가장 복잡한 부분은 텐서 이름 매핑입니다.
아래는 각 컴포넌트별 매핑 구조를 보여줍니다.

In [ ]:
# Diffusion 모델의 텐서 매핑 구조 확인
print("=== Diffusion Model Tensor Mapping Structure ===")
print()
print("Time Embedding:")
print(f"  w1: model.diffusion_model.time_embed.0")
print(f"  w2: model.diffusion_model.time_embed.2")
print()
print("Conv In:  model.diffusion_model.input_blocks.0.0")
print("Conv Out: model.diffusion_model.out.2")
print()
print("Down Encoder Blocks (4 blocks):")
for i in range(4):
    has_transformer = i < 3
    has_downsample = i < 3
    print(f"  Block {i}:")
    for j in range(2):
        idx = i * 3 + j + 1
        print(f"    ResidualBlock {j}: input_blocks.{idx}.0.*")
        if has_transformer:
            print(f"    TransformerBlock {j}: input_blocks.{idx}.1.*")
    if has_downsample:
        print(f"    Downsample: input_blocks.{i*3+3}.0.op")
print()
print("Mid Block:")
print(f"  ResidualBlock 0: middle_block.0.*")
print(f"  TransformerBlock: middle_block.1.*")
print(f"  ResidualBlock 1: middle_block.2.*")
print()
print("Up Decoder Blocks (4 blocks):")
for i in range(4):
    has_transformer = i > 0
    has_upsample = 0 < i < 3
    print(f"  Block {i}:")
    for j in range(3):
        idx = i * 3 + j
        print(f"    ResidualBlock {j}: output_blocks.{idx}.0.*")
        if has_transformer:
            print(f"    TransformerBlock {j}: output_blocks.{idx}.1.*")

## 11. 모바일 배포

### LiteRT Runtime API 직접 사용
C++ 또는 Java/Swift에서 TFLite 런타임으로 직접 모델을 호출할 수 있습니다.
각 컴포넌트(`clip.tflite`, `diffusion.tflite`, `decoder.tflite`)를 순차적으로 호출하는
추론 파이프라인을 구현해야 합니다.

### MediaPipe LLM Inference API 사용
SD 1.5는 LLM이 아니므로 MediaPipe LLM Inference API 대신
LiteRT Runtime API를 직접 사용하는 것이 적합합니다.

### 모델 시각화
```bash
pip install ai-edge-model-explorer
model-explorer diffusion.tflite
```

## 12. 알려진 제한사항

- **메모리 사용량**: 변환 과정에서 가중치의 여러 복사본이 메모리에 유지됩니다. 32GB 이상의 RAM을 권장합니다.
- **VAE Encoder**: 현재 이미지 인코더(img2img용) 변환은 아직 지원되지 않습니다.
- **성능**: XNNPack 백엔드에서 메모리 언패킹이 비효율적일 수 있습니다.
- **GPU 가속**: `device_type="gpu"`로 설정하면 StableHLO composite ops를 활성화하여 GPU 성능을 향상시킬 수 있습니다.

In [ ]:
# GPU 백엔드용 변환 (선택 사항)
# GPU에서 실행하려면 enable_hlfb=True를 사용합니다.

GPU_OUTPUT_DIR = "/tmp/sd15_tflite_gpu"
os.makedirs(GPU_OUTPUT_DIR, exist_ok=True)


@torch.inference_mode()
def convert_for_gpu(ckpt_path: str, output_dir: str):
    """GPU 백엔드 최적화 변환."""
    # GPU 모드로 모델 빌드 (enable_hlfb=True, GroupNorm composite 활성화)
    gpu_diffusion_model = diffusion.Diffusion(
        diffusion.get_model_config(batch_size=2, device_type="gpu")
    )
    gpu_loader = stable_diffusion_loader.DiffusionModelLoader(
        ckpt_path, diffusion.TENSOR_NAMES
    )
    gpu_loader.load(gpu_diffusion_model, strict=False)
    gpu_diffusion_model.eval()

    noise = torch.zeros((1, 4, 64, 64), dtype=torch.float32)
    context = torch.zeros((2, 77, 768), dtype=torch.float32)
    time_emb = util.get_time_embedding(0)

    litert_torch.signature(
        "diffusion",
        gpu_diffusion_model,
        (torch.repeat_interleave(noise, 2, 0), context, time_emb),
    ).convert().export(f"{output_dir}/diffusion_gpu.tflite")

    print(f"GPU-optimized diffusion model saved to: {output_dir}/diffusion_gpu.tflite")


if os.path.exists(CKPT_PATH):
    convert_for_gpu(CKPT_PATH, GPU_OUTPUT_DIR)
else:
    print("Skipping GPU conversion - checkpoint not available.")

## 요약

이 노트북에서 다룬 내용:

1. **Edge Generative API 빌딩 블록**으로 SD 1.5의 CLIP, UNet, VAE Decoder를 재작성
2. **텐서 이름 매핑**을 통해 HuggingFace 체크포인트의 가중치를 로드
3. `litert_torch.signature().convert().export()`로 **TFLite 변환**
4. `quant_recipes`를 사용한 **양자화** (weight-only INT8)
5. 변환된 TFLite 모델로 **이미지 생성 파이프라인** 실행
6. **GPU 백엔드** 최적화 옵션

### 다음 단계
- 다른 양자화 레시피 실험 (INT4, FP16 등)
- Android/iOS 앱에 TFLite 모델 통합
- Model Explorer로 변환된 모델 시각화 및 검증

## 13. 변환된 TFLite 모델로 이미지 생성 Playground

변환이 완료된 TFLite 모델을 사용하여 다양한 프롬프트와 설정으로 이미지를 생성합니다.
프롬프트, 샘플러, CFG 스케일, 스텝 수 등을 자유롭게 변경하여 실험할 수 있습니다.

In [ ]:
# TFLite 파이프라인 로드 (이미 로드되어 있지 않은 경우)
TFLITE_DIR = OUTPUT_DIR  # 양자화 모델을 사용하려면 QUANTIZED_OUTPUT_DIR로 변경

clip_tflite = f"{TFLITE_DIR}/clip.tflite"
diffusion_tflite = f"{TFLITE_DIR}/diffusion.tflite"
decoder_tflite = f"{TFLITE_DIR}/decoder.tflite"

all_models_exist = all(
    os.path.exists(p) for p in [clip_tflite, diffusion_tflite, decoder_tflite]
)

if all_models_exist and os.path.exists(TOKENIZER_DIR):
    if "sd_pipeline" not in dir() or sd_pipeline is None:
        sd_pipeline = StableDiffusionTFLite(
            tokenizer_vocab_dir=TOKENIZER_DIR,
            clip_tflite_path=clip_tflite,
            diffusion_tflite_path=diffusion_tflite,
            decoder_tflite_path=decoder_tflite,
        )
    print("Pipeline ready!")
else:
    print("TFLite models or tokenizer not found. Run sections 5-6 first.")

In [ ]:
# === 이미지 생성 설정 ===
# 아래 값을 자유롭게 변경하여 실험하세요.

PROMPTS = [
    "a beautiful sunset over a calm ocean, oil painting, vivid colors",
    "a cute cat sitting on a windowsill, digital art, detailed fur",
    "futuristic cyberpunk cityscape at night, neon lights, rain",
    "a cozy cabin in snowy mountains, watercolor style",
]

NEGATIVE_PROMPT = "blurry, low quality, deformed, ugly, bad anatomy"
CFG_SCALE = 7.5
SAMPLER = "k_euler"          # "k_euler", "k_euler_ancestral", "k_lms"
N_STEPS = 20                 # 추론 스텝 수 (높을수록 품질↑, 속도↓)
SEED = 42                    # None으로 설정하면 랜덤

print(f"Prompts: {len(PROMPTS)}")
print(f"Negative prompt: {NEGATIVE_PROMPT}")
print(f"CFG scale: {CFG_SCALE}, Sampler: {SAMPLER}, Steps: {N_STEPS}, Seed: {SEED}")

In [ ]:
# 여러 프롬프트로 이미지 일괄 생성
import time

generated_images = []
generation_times = []

if "sd_pipeline" in dir() and sd_pipeline is not None:
    for i, prompt in enumerate(PROMPTS):
        print(f"\n{'='*60}")
        print(f"[{i+1}/{len(PROMPTS)}] \"{prompt}\"")
        print(f"{'='*60}")

        output_path = f"{OUTPUT_DIR}/playground_{i:02d}.png"
        start_time = time.time()

        img = generate_image(
            model=sd_pipeline,
            prompt=prompt,
            output_path=output_path,
            uncond_prompt=NEGATIVE_PROMPT,
            cfg_scale=CFG_SCALE,
            sampler_name=SAMPLER,
            n_inference_steps=N_STEPS,
            seed=SEED + i if SEED is not None else None,
        )
        elapsed = time.time() - start_time
        generation_times.append(elapsed)
        generated_images.append(img)

        print(f"  Time: {elapsed:.1f}s")

    print(f"\nAll {len(PROMPTS)} images generated!")
    print(f"Average time per image: {np.mean(generation_times):.1f}s")
else:
    print("Pipeline not loaded. Run the previous cell first.")

In [ ]:
# 생성된 이미지를 그리드로 표시
from IPython.display import display
import math

if generated_images:
    n_images = len(generated_images)
    cols = min(n_images, 2)
    rows = math.ceil(n_images / cols)

    grid_w = cols * 512
    grid_h = rows * 512
    grid = Image.new("RGB", (grid_w, grid_h), color=(255, 255, 255))

    for idx, img in enumerate(generated_images):
        r, c = divmod(idx, cols)
        grid.paste(img.resize((512, 512)), (c * 512, r * 512))

    # 프롬프트 목록 출력
    for i, prompt in enumerate(PROMPTS):
        print(f"[{i}] {prompt}")
    print()

    display(grid)
    grid.save(f"{OUTPUT_DIR}/playground_grid.png")
    print(f"\nGrid saved to: {OUTPUT_DIR}/playground_grid.png")
else:
    print("No images generated yet.")

### 단일 이미지 생성 (빠른 테스트)

아래 셀에서 프롬프트를 직접 변경하여 단일 이미지를 빠르게 생성할 수 있습니다.

In [ ]:
# === 단일 이미지 생성 ===
# 프롬프트를 변경하고 이 셀을 실행하세요.

MY_PROMPT = "a magical forest with glowing mushrooms, fantasy art, ethereal lighting"
MY_NEGATIVE = "blurry, low quality, deformed"

if "sd_pipeline" in dir() and sd_pipeline is not None:
    start_time = time.time()
    single_image = generate_image(
        model=sd_pipeline,
        prompt=MY_PROMPT,
        output_path=f"{OUTPUT_DIR}/single_output.png",
        uncond_prompt=MY_NEGATIVE,
        cfg_scale=7.5,
        sampler_name="k_euler",
        n_inference_steps=20,
        seed=123,
    )
    print(f"Time: {time.time() - start_time:.1f}s")
    display(single_image)
else:
    print("Pipeline not loaded.")

### CFG Scale 비교

CFG(Classifier-Free Guidance) 스케일 값에 따른 이미지 변화를 비교합니다.
- **낮은 값 (1-3)**: 프롬프트를 느슨하게 따름, 더 다양한 결과
- **중간 값 (7-8)**: 일반적으로 가장 좋은 품질
- **높은 값 (15+)**: 프롬프트에 강하게 집착, 과포화 가능

In [ ]:
# CFG Scale 비교 실험
CFG_PROMPT = "a beautiful landscape with mountains and a lake, photorealistic"
CFG_VALUES = [2.0, 7.5, 15.0]

cfg_images = []

if "sd_pipeline" in dir() and sd_pipeline is not None:
    for cfg_val in CFG_VALUES:
        print(f"\nGenerating with CFG scale = {cfg_val}...")
        img = generate_image(
            model=sd_pipeline,
            prompt=CFG_PROMPT,
            output_path=f"{OUTPUT_DIR}/cfg_{cfg_val:.1f}.png",
            uncond_prompt="blurry, low quality",
            cfg_scale=cfg_val,
            sampler_name="k_euler",
            n_inference_steps=20,
            seed=42,
        )
        cfg_images.append(img)

    # 가로로 나란히 표시
    total_w = 512 * len(cfg_images)
    comparison = Image.new("RGB", (total_w, 512))
    for i, img in enumerate(cfg_images):
        comparison.paste(img.resize((512, 512)), (i * 512, 0))

    print(f"\nCFG Scale: {' | '.join(str(v) for v in CFG_VALUES)}")
    display(comparison)
    comparison.save(f"{OUTPUT_DIR}/cfg_comparison.png")
else:
    print("Pipeline not loaded.")

### Sampler 비교

동일한 프롬프트와 시드로 다른 샘플러를 비교합니다.
- **k_euler**: 빠르고 안정적, 일반적인 선택
- **k_euler_ancestral**: 더 다양한 결과, 약간의 랜덤성 추가
- **k_lms**: Linear Multistep, 부드러운 결과

In [ ]:
# Sampler 비교 실험
SAMPLER_PROMPT = "a portrait of a wise old wizard, fantasy art, detailed"
SAMPLERS_TO_COMPARE = ["k_euler", "k_euler_ancestral", "k_lms"]

sampler_images = []

if "sd_pipeline" in dir() and sd_pipeline is not None:
    for sampler_name in SAMPLERS_TO_COMPARE:
        print(f"\nGenerating with sampler = {sampler_name}...")
        img = generate_image(
            model=sd_pipeline,
            prompt=SAMPLER_PROMPT,
            output_path=f"{OUTPUT_DIR}/sampler_{sampler_name}.png",
            uncond_prompt="blurry, low quality",
            cfg_scale=7.5,
            sampler_name=sampler_name,
            n_inference_steps=20,
            seed=42,
        )
        sampler_images.append(img)

    # 가로로 나란히 표시
    total_w = 512 * len(sampler_images)
    comparison = Image.new("RGB", (total_w, 512))
    for i, img in enumerate(sampler_images):
        comparison.paste(img.resize((512, 512)), (i * 512, 0))

    print(f"\nSamplers: {' | '.join(SAMPLERS_TO_COMPARE)}")
    display(comparison)
    comparison.save(f"{OUTPUT_DIR}/sampler_comparison.png")
else:
    print("Pipeline not loaded.")